# BudgetCE — reproduce the results
**Irtiqa Haider** · https://github.com/IrtiqaHaider/budgetce

Default: verify and analyze the included records on CPU. Optional cells reproduce the GPU experiment on a fresh runtime. No LaTeX tools or pretrained weights are needed. The repository must be published before its clone/Colab link works; a local extracted repository also works. The report and measurements have bounded T4 scope, not universal speed guarantees.

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPOSITORY = "https://github.com/IrtiqaHaider/budgetce.git"
RUN_NEW_EXPERIMENT = False  # True explicitly enables new GPU measurements.
PERSIST_TO_DRIVE = False
RUN_NAME = "reproduction_001"

candidates = [Path.cwd(), Path.cwd().parent, Path("/content/budgetce")]
PROJECT = next((p for p in candidates if (p / "budgetce/ops.py").is_file()), None)
if PROJECT is None:
    PROJECT = Path("/content/budgetce") if Path("/content").exists() else Path.cwd() / "budgetce"
    subprocess.run(["git", "clone", "--depth", "1", REPOSITORY, str(PROJECT)], check=True)
PROJECT = PROJECT.resolve()
print("Repository:", PROJECT)

## 1 — Supporting dependencies
The requirements deliberately do not install or replace PyTorch. For new GPU execution, keep a compatible CUDA-enabled build and toolkit.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT / "requirements.txt")], check=True)

def run(args, **kwargs):
    env = dict(os.environ, PYTHONPATH=str(PROJECT), OMP_NUM_THREADS="2", MKL_NUM_THREADS="2", MAX_JOBS="1")
    return subprocess.run([sys.executable, *args], cwd=PROJECT, env=env, check=True, **kwargs)

run(["scripts/verify_release.py"])
run(["scripts/reproduce.py", "--out", str(PROJECT / "reanalysis")])
run(["scripts/plot_results.py", "--results", str(PROJECT / "reanalysis")])

## 2 — Inspect the recorded results
The initial and matched-window results are separate. Quantities in these tables come from archived raw measurements, not from new inference or training.

In [ ]:
import pandas as pd
from IPython.display import display, Image

display(pd.read_csv(PROJECT / "reanalysis/matched_training.csv"))
display(pd.read_csv(PROJECT / "reanalysis/operator_comparisons.csv"))
display(Image(filename=str(PROJECT / "reanalysis/figures/matched_training.png")))
run(["-m", "pytest", "-q"])

## 3 — Optional: new GPU run
Only runs when `RUN_NEW_EXPERIMENT = True`. This does not append to the published results. The original 21-check CUDA gate must pass. Do not reduce tolerances to obtain measurements. A fresh environment is recorded, and incomplete/failed trials remain visible. Compilation and GPU execution of this repository wrapper were not repeated during packaging.

In [ ]:
if RUN_NEW_EXPERIMENT:
    import torch
    from torch.utils.cpp_extension import CUDA_HOME
    if not torch.cuda.is_available():
        raise RuntimeError("Choose a Colab NVIDIA GPU runtime before running new measurements.")
    if CUDA_HOME is None or not (Path(CUDA_HOME) / "bin/nvcc").is_file():
        raise RuntimeError("A compatible CUDA toolkit with nvcc is required.")
    if torch.cuda.get_device_capability(0) < (7, 5):
        raise RuntimeError("This implementation targets compute capability 7.5 or newer.")
    print(torch.cuda.get_device_name(0), torch.__version__)
    if PERSIST_TO_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        base = Path("/content/drive/MyDrive/budgetce_runs")
    else:
        base = PROJECT / "runs"
    OUT = base / RUN_NAME
    (OUT / "validation").mkdir(parents=True, exist_ok=True)
    tests = ["tests/test_artifact.py", "tests/test_dispatch.py", "tests/test_model.py", "tests/test_ops.py", "tests/test_planner.py"]
    result = run(["-m", "pytest", "-q", *tests], capture_output=True, text=True)
    (OUT / "validation/cpu-tests.txt").write_text(result.stdout + result.stderr)
    run(["-m", "budgetce.cli", "init", "--out", str(OUT), "--config", str(PROJECT / "configs/confirm.json")])
    run(["-m", "budgetce.cli", "validate", "--out", str(OUT)])
else:
    print("New GPU execution is disabled; the recorded-data audit is complete.")

## 4 — Optional: original operator and short-training study
20 calibration blocks, 90 held-out operator blocks and 18 short training trials. The analysis exports actual failures; it does not silently retry until success.

In [ ]:
if RUN_NEW_EXPERIMENT:
    try:
        for stage in ["operators", "train", "analyze"]:
            run(["-m", "budgetce.cli", stage, "--out", str(OUT)])
    finally:
        run(["-m", "budgetce.cli", "export", "--out", str(OUT)])

## 5 — Optional: longer matched-chunk confirmation
Nine trials: S=1,024; native, PyTorch:1024 and CUDA:1024; 20 warmup plus 100 measured updates. The runner writes and verifies two additional numerical checks. CUDA selection is fixed and is not credited to the learned policy.

In [ ]:
if RUN_NEW_EXPERIMENT:
    run(["scripts/confirm9.py", "--prior", str(OUT)])
    print("New archives are in:", OUT.parent)
    print("Keep new results separate from the recorded evidence and report.")

## 6 — Optional: download the new confirmation archive
The most recently modified confirmation ZIP in this run directory is selected below. No new GPU archive is created by CPU-only auditing.

In [ ]:
if RUN_NEW_EXPERIMENT:
    archives = sorted(OUT.parent.glob("budgetce_matched9_*_results.zip"), key=lambda p: p.stat().st_mtime)
    if archives:
        print("Confirmation archive:", archives[-1])
        try:
            from google.colab import files
            files.download(str(archives[-1]))
        except ImportError:
            pass